SyntaxError: invalid character '→' (U+2192) (4123413212.py, line 25)

# HCLProject
Travel Reimbursement Approval Agent 

# Travel Reimbursement Approval Agent

A lightweight agentic prototype that reviews travel claims against a mock policy (Appendix A) and returns a structured decision for the five sample claims (Appendix B).

## README

**Setup**
```bash
pip install pandas matplotlib            # required
pip install anthropic                    # optional, only for LLM mode
jupyter notebook abhishek.ipynb          # or: jupyter nbconvert --to notebook --execute abhishek.ipynb
```

**Environment variables (all optional)**
| Variable | Purpose |
|---|---|
| `ANTHROPIC_API_KEY` | If set (and `anthropic` is installed) the agent runs in **LLM mode**: the model chooses which tools to call and writes the explanation. |
| `CLAIM_AGENT_MODEL` | Model name for LLM mode (default `claude-sonnet-5-5`). |

Without a key the notebook runs in **offline mode**: a scripted planner drives the *same* tools through the *same* loop, so the notebook runs top-to-bottom with no manual steps.

**Run:** Kernel → Restart & Run All. The last code cell prints the final JSON array.

**Key design choices**
1. LLM proposes, deterministic policy engine disposes: money math and the final decision come from tested tool code, never from free-text LLM arithmetic.
2. Every decision is validated (`validate_output`) and falls back to MANUAL_REVIEW if anything is inconsistent.
3. Uncertain, incomplete or exception cases go to MANUAL_REVIEW rather than being forced.


# HCLProject
Travel Reimbursement Approval Agent 

# Travel Reimbursement Approval Agent

A lightweight agentic prototype that reviews travel claims against a mock policy (Appendix A) and returns a structured decision for the five sample claims (Appendix B).

## README

**Setup**
```bash
pip install pandas matplotlib            # required
pip install anthropic                    # optional, only for LLM mode
jupyter notebook abhishek.ipynb          # or: jupyter nbconvert --to notebook --execute abhishek.ipynb
```

**Environment variables (all optional)**
| Variable | Purpose |
|---|---|
| `ANTHROPIC_API_KEY` | If set (and `anthropic` is installed) the agent runs in **LLM mode**: the model chooses which tools to call and writes the explanation. |
| `CLAIM_AGENT_MODEL` | Model name for LLM mode (default `claude-sonnet-5-5`). |

Without a key the notebook runs in **offline mode**: a scripted planner drives the *same* tools through the *same* loop, so the notebook runs top-to-bottom with no manual steps.

**Run:** Kernel → Restart & Run All. The last code cell prints the final JSON array.

**Key design choices**
1. LLM proposes, deterministic policy engine disposes: money math and the final decision come from tested tool code, never from free-text LLM arithmetic.
2. Every decision is validated (`validate_output`) and falls back to MANUAL_REVIEW if anything is inconsistent.
3. Uncertain, incomplete or exception cases go to MANUAL_REVIEW rather than being forced.


## 1. Policy & limits (Appendix A as data)


In [5]:
import os, re, json, datetime as dt
from collections import Counter
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

POLICY = {
 "POL-CAT-01": "Eligible categories: airfare (economy only), lodging, meals (per-diem), ground transport (taxi, rideshare, train, rental car, parking), conference / registration fees. Reimbursable when incurred for a documented business purpose.",
 "POL-CAT-02": "Ineligible items, never reimbursable, rejected and deducted in full: alcohol, minibar, spa, gym, personal entertainment, in-room movies, personal shopping, gifts, traffic fines, penalties, late fees, any personal expense.",
 "POL-PD-01":  "Meals maximum $75 per day. Amounts above the daily cap are deducted; the rest is reimbursed.",
 "POL-PD-02":  "Lodging maximum $200 per night. Amounts above the nightly cap are deducted; the rest is reimbursed.",
 "POL-PD-03":  "Ground transport maximum $50 per day. Amounts above the cap are deducted.",
 "POL-AIR-01": "Airfare class: only economy class airfare is reimbursable. Business or first class is a policy exception and must be routed to Manual Review, not auto-deducted, because a pre-approval may exist.",
 "POL-RCT-01": "Receipt required above $25: any single line item greater than $25 requires an itemized receipt. Airfare and lodging always require a receipt regardless of amount.",
 "POL-RCT-02": "Missing receipt handling: not silently rejected; the claim is routed to Manual Review so the reviewer can request the receipt.",
 "POL-APR-01": "Auto-approve tier: total reimbursable <= $500 may be auto-approved by the agent if fully compliant.",
 "POL-APR-02": "Manager tier: total > $500 and <= $2,000 is eligible for approval, treated as approvable when fully compliant.",
 "POL-APR-03": "Director / Manual-Review tier: total > $2,000 exceeds agent authority and must be routed to Manual Review (director approval), even if otherwise compliant.",
 "POL-TIME-01":"Submission window: claims must be submitted within 30 days of the expense date. Late claims are routed to Manual Review.",
}
DAILY_CAPS = {"meals": ("POL-PD-01", 75.0), "lodging": ("POL-PD-02", 200.0), "ground_transport": ("POL-PD-03", 50.0)}
INELIGIBLE = {"spa", "minibar", "alcohol", "gym", "entertainment", "movies", "shopping", "gifts", "fines", "penalties", "late_fees", "personal"}
ELIGIBLE = {"airfare", "lodging", "meals", "ground_transport", "conference_fees"}
ALWAYS_RECEIPT = {"airfare", "lodging"}
RECEIPT_THRESHOLD = 25.0
SUBMISSION_WINDOW_DAYS = 30

## 2. Claim intake (Appendix B as JSON)
`qty` is the number of nights/days the line covers, taken from the claim description (e.g. "2 nights @ $180"). It is needed to apply per-day / per-night caps.

In [6]:
CLAIMS = [
 {"claim_id":"CLM-001","employee":"A. Rivera","purpose":"Attend 2-day industry conference (business)","trip_start":"2026-06-10","trip_end":"2026-06-12","submitted":"2026-06-20","items":[
   {"category":"airfare","description":"Round-trip economy airfare","amount":420.00,"receipt":True,"qty":None},
   {"category":"lodging","description":"Hotel, 2 nights @ $180","amount":360.00,"receipt":True,"qty":2},
   {"category":"meals","description":"Meals, 3 days @ ~$60/day","amount":180.00,"receipt":True,"qty":3},
   {"category":"conference_fees","description":"Conference registration","amount":150.00,"receipt":True,"qty":None}]},
 {"claim_id":"CLM-002","employee":"B. Osei","purpose":"Weekend hotel stay","trip_start":"2026-06-14","trip_end":"2026-06-15","submitted":"2026-06-25","items":[
   {"category":"spa","description":"Hotel spa package","amount":300.00,"receipt":True,"qty":None},
   {"category":"minibar","description":"In-room minibar","amount":80.00,"receipt":True,"qty":None}]},
 {"claim_id":"CLM-003","employee":"C. Nakamura","purpose":"Client site visit (business)","trip_start":"2026-06-08","trip_end":"2026-06-10","submitted":"2026-06-22","items":[
   {"category":"airfare","description":"Round-trip economy airfare","amount":300.00,"receipt":True,"qty":None},
   {"category":"lodging","description":"Hotel, 2 nights @ $250","amount":500.00,"receipt":True,"qty":2},
   {"category":"meals","description":"Meals, 2 days @ $70/day","amount":140.00,"receipt":True,"qty":2}]},
 {"claim_id":"CLM-004","employee":"D. Fischer","purpose":"International vendor negotiation (business)","trip_start":"2026-06-16","trip_end":"2026-06-18","submitted":"2026-06-28","items":[
   {"category":"airfare","description":"Business-class international airfare","amount":2400.00,"receipt":True,"qty":None},
   {"category":"lodging","description":"Hotel, 3 nights","amount":600.00,"receipt":False,"qty":3}]},
 {"claim_id":"CLM-005","employee":"E. Haddad","purpose":"Client dinner / business development","trip_start":"2026-06-11","trip_end":"2026-06-11","submitted":"2026-06-24","items":[
   {"category":"meals","description":"Client dinner for 4 (business development)","amount":220.00,"receipt":False,"qty":1}]},
]
pd.DataFrame([{"claim_id":c["claim_id"],"employee":c["employee"],"items":len(c["items"]),"claimed":sum(i["amount"] for i in c["items"])} for c in CLAIMS])

,claim_id,employee,items,claimed
0,CLM-001,A. Rivera,4,1110.0
1,CLM-002,B. Osei,2,380.0
2,CLM-003,C. Nakamura,3,940.0
3,CLM-004,D. Fischer,2,3000.0
4,CLM-005,E. Haddad,1,220.0


## 3. Tools
Each tool is a small, pure function that returns a JSON-serialisable dict, including the `policy_refs` it relied on. `policy_lookup` grounds the agent in policy text (a keyword retriever is enough for 12 rules; see Design Notes for when to switch to embeddings).

In [7]:
def _find(claim_id):
    return next(c for c in CLAIMS if c["claim_id"] == claim_id)

def policy_lookup(query: str, k: int = 3):
    """Retrieve the k most relevant policy rules by keyword overlap."""
    q = set(re.findall(r"[a-z0-9\-]+", query.lower()))
    scored = []
    for rid, text in POLICY.items():
        words = set(re.findall(r"[a-z0-9\-]+", (rid + " " + text).lower()))
        scored.append((len(q & words), rid))
    scored.sort(reverse=True)
    return {"rules": [{"id": r, "text": POLICY[r]} for s, r in scored[:k] if s > 0]}

def check_eligibility(claim_id: str):
    """Split line items into eligible / ineligible / unknown categories."""
    c = _find(claim_id); inel, el, unk = [], [], []
    for it in c["items"]:
        cat = it["category"]
        (inel if cat in INELIGIBLE else el if cat in ELIGIBLE else unk).append(it)
    refs = (["POL-CAT-02"] if inel else []) + (["POL-CAT-01"] if el else [])
    return {"ineligible": [(i["category"], i["amount"]) for i in inel],
            "eligible": [(i["category"], i["amount"]) for i in el],
            "unknown": [(i["category"], i["amount"]) for i in unk],
            "ineligible_total": sum(i["amount"] for i in inel), "policy_refs": refs}

def check_receipts(claim_id: str):
    """Find eligible items that need a receipt but have none (ineligible items are ignored: they are rejected anyway)."""
    c = _find(claim_id); missing = []
    for it in c["items"]:
        if it["category"] not in ELIGIBLE: continue
        needs = it["category"] in ALWAYS_RECEIPT or it["amount"] > RECEIPT_THRESHOLD
        if needs and not it["receipt"]:
            missing.append(f"{it['category']} receipt (${it['amount']:.2f})")
    return {"missing_docs": missing, "policy_refs": ["POL-RCT-01"] + (["POL-RCT-02"] if missing else [])}

def check_limits(claim_id: str):
    """Apply per-day / per-night caps to eligible items and flag airfare class exceptions."""
    c = _find(claim_id); lines, flags, refs = [], [], set()
    for it in c["items"]:
        cat = it["category"]
        if cat not in ELIGIBLE: continue
        allowed, ded = it["amount"], 0.0
        if cat in DAILY_CAPS:
            ref, cap = DAILY_CAPS[cat]; refs.add(ref)
            units = it["qty"] or 1
            allowed = min(it["amount"], cap * units); ded = round(it["amount"] - allowed, 2)
            if ded > 0 and cat == "meals" and re.search(r"for\s+\d+|group|clients?", it["description"].lower()):
                flags.append(f"Meal '{it['description']}' may cover several people; caps are per day, so per-head treatment is ambiguous")
        if cat == "airfare":
            refs.add("POL-AIR-01")
            if re.search(r"business|first", it["description"].lower()):
                flags.append(f"Non-economy airfare (${it['amount']:.2f}) is a policy exception: possible pre-approval")
        lines.append({"category": cat, "claimed": it["amount"], "allowed": allowed, "deducted": ded})
    return {"lines": lines, "per_diem_deduction": round(sum(l["deducted"] for l in lines), 2), "exception_flags": flags, "policy_refs": sorted(refs)}

def check_timeliness(claim_id: str):
    """Submission must be within 30 days of the expense date (earliest trip date used, conservatively)."""
    c = _find(claim_id)
    days = (dt.date.fromisoformat(c["submitted"]) - dt.date.fromisoformat(c["trip_start"])).days
    return {"days_since_expense": days, "late": days > SUBMISSION_WINDOW_DAYS, "policy_refs": ["POL-TIME-01"]}

def check_approval_threshold(total_reimbursable: float):
    """Map reimbursable total to an approval tier."""
    if total_reimbursable <= 500:  tier, ref, auto = "auto", "POL-APR-01", True
    elif total_reimbursable <= 2000: tier, ref, auto = "manager", "POL-APR-02", True
    else: tier, ref, auto = "director", "POL-APR-03", False
    return {"tier": tier, "agent_may_approve": auto, "policy_refs": [ref]}

TOOLS = {"policy_lookup": policy_lookup, "check_eligibility": check_eligibility, "check_receipts": check_receipts,
         "check_limits": check_limits, "check_timeliness": check_timeliness, "check_approval_threshold": check_approval_threshold}

## 4. Decision engine and output validator
`decide()` combines tool results using the *Decision Guidance* rules. Precedence: any exception, ambiguity, missing receipt, late claim or high value → **MANUAL_REVIEW**; otherwise nothing reimbursable → **REJECT**; otherwise any deduction → **PARTIAL_APPROVE**; otherwise **APPROVE**.

For MANUAL_REVIEW, `approved_amount` and `deducted_amount` are both `0` because no money is committed until the reviewer rules; the potential amounts are quoted in the explanation.

In [8]:
DECISIONS = {"APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"}
FIELDS = ["claim_id","decision","approved_amount","deducted_amount","missing_docs","policy_refs","confidence","explanation","tools_used"]

def decide(claim, r):
    """Pure function: tool results -> decision. r maps tool name -> result."""
    total = round(sum(i["amount"] for i in claim["items"]), 2)
    el, lim, rc, tm = r["check_eligibility"], r.get("check_limits"), r.get("check_receipts"), r.get("check_timeliness")
    ineligible = round(el["ineligible_total"], 2)
    reimbursable = round(sum(l["allowed"] for l in lim["lines"]), 2) if lim else 0.0
    per_diem = lim["per_diem_deduction"] if lim else 0.0
    reasons, flags = [], []
    if el["unknown"]: flags.append("unrecognised category: " + ", ".join(c for c, _ in el["unknown"]))
    if lim and lim["exception_flags"]: flags += lim["exception_flags"]
    if rc and rc["missing_docs"]: flags.append("missing required receipt(s): " + "; ".join(rc["missing_docs"]))
    if tm and tm["late"]: flags.append(f"submitted {tm['days_since_expense']} days after expense (> 30)")
    tier = check_approval_threshold(reimbursable) if lim else None
    if tier and not tier["agent_may_approve"]: flags.append(f"reimbursable total ${reimbursable:,.2f} exceeds $2,000 agent authority (director approval)")
    refs = set(el["policy_refs"]) | set(lim["policy_refs"] if lim else []) | set(rc["policy_refs"] if rc else []) | set(tm["policy_refs"] if tm else []) | set(tier["policy_refs"] if tier else [])

    if flags:
        decision, approved, deducted = "MANUAL_REVIEW", 0.0, 0.0
        exposure = f" If approved as claimed, per-diem caps would deduct ${per_diem:,.2f}." if per_diem else ""
        expl = (f"Routed to manual review: {'; '.join(flags)}. ${total:,.2f} is held pending reviewer action "
                f"(${reimbursable:,.2f} otherwise reimbursable, ${ineligible:,.2f} ineligible).{exposure}")
        conf = round(max(0.6, 0.9 - 0.05 * len(flags)), 2)
    elif reimbursable == 0:
        decision, approved, deducted = "REJECT", 0.0, total
        expl = f"Rejected: every item is ineligible under POL-CAT-02 ({', '.join(c for c, _ in el['ineligible'])}); ${total:,.2f} deducted in full."
        conf = 0.97
    elif per_diem > 0 or ineligible > 0:
        decision, approved, deducted = "PARTIAL_APPROVE", reimbursable, round(per_diem + ineligible, 2)
        parts = [f"{l['category']} capped at ${l['allowed']:,.2f} (claimed ${l['claimed']:,.2f})" for l in lim["lines"] if l["deducted"] > 0]
        if ineligible: parts.append(f"${ineligible:,.2f} ineligible items removed")
        expl = f"Partially approved: ${approved:,.2f} reimbursed, ${deducted:,.2f} deducted. " + "; ".join(parts) + f". Total is in the {tier['tier']} tier."
        conf = 0.92
    else:
        decision, approved, deducted = "APPROVE", reimbursable, 0.0
        expl = f"Approved in full: all items eligible, receipts present, within per-diem caps, submitted on time, ${approved:,.2f} falls in the {tier['tier']} tier."
        conf = 0.95
    return {"decision": decision, "approved_amount": approved, "deducted_amount": deducted,
            "missing_docs": rc["missing_docs"] if rc else [], "policy_refs": sorted(refs), "confidence": conf, "explanation": expl}

def validate_output(claim_id: str, out: dict):
    """Schema + arithmetic checks. Returns a list of problems (empty == valid)."""
    claim, errs = _find(claim_id), []
    miss = [f for f in FIELDS if f not in out]
    if miss: errs.append(f"missing fields {miss}")
    if out.get("decision") not in DECISIONS: errs.append("invalid decision")
    if not 0 <= out.get("confidence", -1) <= 1: errs.append("confidence out of range")
    total = round(sum(i["amount"] for i in claim["items"]), 2)
    if out.get("decision") in {"APPROVE", "PARTIAL_APPROVE", "REJECT"} and abs(out.get("approved_amount", 0) + out.get("deducted_amount", 0) - total) > 0.01:
        errs.append("approved + deducted != claimed total")
    if out.get("decision") == "APPROVE" and out.get("missing_docs"): errs.append("APPROVE with missing docs")
    return errs
TOOLS["validate_output"] = validate_output

## 5. Agent loop
Two interchangeable planners share one executor, one audit trail and one validator:

* **LLM mode**: the model is given the tool schemas and decides which to call, in what order, and what to do with conflicting results. Its own proposed decision is *compared* with the policy engine; any disagreement or invalid output is overridden and flagged in the audit trail.
* **Offline mode**: a scripted planner that mimics the same conditional behaviour (e.g. it skips limit/receipt checks when nothing is reimbursable). It exists so the notebook is reproducible without credentials.

In [9]:
import os
AUDIT = {}   # claim_id -> list of steps (audit trail)

def _log(cid, **kw): AUDIT.setdefault(cid, []).append(kw)

def run_tool(cid, name, **args):
    res = TOOLS[name](**args) if name != "validate_output" else TOOLS[name](cid, **args)
    _log(cid, step="tool_call", tool=name, args=args, result=res)
    return res

def offline_planner(claim):
    """Scripted stand-in for the LLM: chooses tools conditionally and combines results."""
    cid, r = claim["claim_id"], {}
    r["policy_lookup"] = run_tool(cid, "policy_lookup", query=claim["purpose"] + " " + " ".join(i["category"] for i in claim["items"]))
    r["check_eligibility"] = run_tool(cid, "check_eligibility", claim_id=cid)
    if r["check_eligibility"]["eligible"] or r["check_eligibility"]["unknown"]:
        r["check_receipts"] = run_tool(cid, "check_receipts", claim_id=cid)
        r["check_limits"] = run_tool(cid, "check_limits", claim_id=cid)
        r["check_timeliness"] = run_tool(cid, "check_timeliness", claim_id=cid)
        run_tool(cid, "check_approval_threshold", total_reimbursable=sum(l["allowed"] for l in r["check_limits"]["lines"]))
    else:
        _log(cid, step="skip", note="nothing eligible, so receipt/limit/threshold checks are not needed")
    return r

TOOL_SCHEMAS = [
 {"name": "policy_lookup", "description": "Retrieve relevant policy rules by keyword.", "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
 {"name": "check_eligibility", "description": "Classify line items as eligible/ineligible.", "input_schema": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
 {"name": "check_receipts", "description": "List required receipts that are missing.", "input_schema": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
 {"name": "check_limits", "description": "Apply per-diem caps and detect airfare class exceptions.", "input_schema": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
 {"name": "check_timeliness", "description": "Check the 30-day submission window.", "input_schema": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}},
 {"name": "check_approval_threshold", "description": "Approval tier for a reimbursable total.", "input_schema": {"type": "object", "properties": {"total_reimbursable": {"type": "number"}}, "required": ["total_reimbursable"]}},
]
SYSTEM = ("You are a travel-reimbursement review agent. Use the tools to check the claim against policy. Never do money arithmetic yourself; rely on tool results. "
          "Prefer MANUAL_REVIEW when anything is ambiguous, missing or an exception. Finish with one line: DECISION=<APPROVE|PARTIAL_APPROVE|REJECT|MANUAL_REVIEW> then a 1-2 sentence explanation citing POL ids.")

def llm_planner(claim, client, model):
    """Tool-calling loop with the Anthropic API. Returns (tool results, LLM's proposed decision)."""
    cid, r = claim["claim_id"], {}
    msgs = [{"role": "user", "content": "Review this claim:\n" + json.dumps(claim)}]
    for _ in range(10):
        resp = client.messages.create(model=model, max_tokens=1000, system=SYSTEM, tools=TOOL_SCHEMAS, messages=msgs)
        msgs.append({"role": "assistant", "content": resp.content})
        calls = [b for b in resp.content if b.type == "tool_use"]
        if not calls:
            text = "".join(b.text for b in resp.content if b.type == "text")
            _log(cid, step="llm_final", text=text)
            m = re.search(r"DECISION=(\w+)", text)
            return r, (m.group(1) if m else None)
        results = []
        for b in calls:
            out = run_tool(cid, b.name, **b.input); r[b.name] = out
            results.append({"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(out)})
        msgs.append({"role": "user", "content": results})
    return r, None

def get_client():
    try:
        import anthropic
        if os.environ.get("fdis_01W5GcaBaE4QbFxAHVQnAW2M"): return anthropic.Anthropic()
    except ImportError:
        pass
    return None

def review_claim(claim, mode="auto"):
    cid, client = claim["claim_id"], get_client()
    AUDIT[cid] = []
    llm_decision = None
    if mode in ("auto", "llm") and client:
        try:
            r, llm_decision = llm_planner(claim, client, os.environ.get("CLAIM_AGENT_MODEL", "claude-sonnet-5-5"))
            for need in ("check_eligibility", "check_receipts", "check_limits", "check_timeliness"):   # guarantee required evidence
                if need not in r: r[need] = run_tool(cid, need, claim_id=cid)
            _log(cid, step="mode", value="llm")
        except Exception as e:                                   # API/network failure -> fallback, never crash
            _log(cid, step="fallback", reason=repr(e)); AUDIT[cid] = AUDIT[cid][-1:]
            r = offline_planner(claim)
    else:
        r = offline_planner(claim); _log(cid, step="mode", value="offline")
    out = decide(claim, r)
    if llm_decision and llm_decision != out["decision"]:
        _log(cid, step="override", llm=llm_decision, engine=out["decision"], note="engine decision wins; flagged for audit")
        out["confidence"] = min(out["confidence"], 0.7)
    errs = run_tool(cid, "validate_output", out={**out, "claim_id": cid, "tools_used": []})
    errs = [e for e in errs if e != "missing fields ['tools_used']"]
    if errs:   # validation failure -> safe fallback
        out.update(decision="MANUAL_REVIEW", approved_amount=0.0, deducted_amount=0.0, confidence=0.5,
                   explanation="Output failed validation (" + "; ".join(errs) + "); routed to manual review.")
    tools = list(dict.fromkeys(s["tool"] for s in AUDIT[cid] if s["step"] == "tool_call"))
    return {"claim_id": cid, **out, "tools_used": tools}

print("mode:", "LLM (API key found)" if get_client() else "offline (no ANTHROPIC_API_KEY)")

mode: offline (no ANTHROPIC_API_KEY)
